# Part 1: Descriptive Analytics

## 1.1 Dataset Introduction
The Ames Housing dataset contains 2,930 residential property sales in Ames, Iowa, with 82 columns detailing physical characteristics of the homes (such as lot size, above-ground living area, number of rooms, garage/basement details) as well as condition/quality ratings, neighborhood information, and sale specifics.

The target variable for our predictive modeling is **SalePrice** (continuous, in USD). 

**Why this is interesting:** Predicting home prices is crucial for multiple stakeholders. It helps buyers to avoid overpaying, sellers to set competitive listing prices, and real estate professionals to advise their clients effectively. Understanding precisely which features (e.g., overall quality, square footage, neighborhood) most significantly drive a home's value can also empower homeowners to make cost-effective renovation and home improvement decisions with the highest ROI.

**Basic Statistics:** The raw dataset comprises 2,930 rows and 82 columns (39 numerical and 43 categorical features).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
import shap

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

df = pd.read_csv("AmesHousing.csv")
print(f"Dataset shape: {df.shape}")
print(f"SalePrice Range: ${df['SalePrice'].min():,.0f} to ${df['SalePrice'].max():,.0f}")
print(f"SalePrice Mean: ${df['SalePrice'].mean():,.0f}")

## 1.2 Target Distribution
The plot below shows the distribution of our target variable, `SalePrice`. The distribution is clearly **right-skewed** with a long tail of very expensive properties. Most homes sell between $100K and $250K, but there are notable outliers above $500K. Because of this right skewness, a log transformation of the SalePrice may be beneficial for linear models to help normalize the variance, though tree-based models generally handle it well.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['SalePrice'], kde=True, bins=50, color='blue')
plt.title('Distribution of SalePrice', fontsize=14)
plt.xlabel('Sale Price ($)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('results/target_dist.png')
plt.show()

## 1.3 Feature Distributions and Relationships
Exploring how different features relate to the `SalePrice`.

In [ ]:
# 1. Boxplot of SalePrice by Overall Quality
plt.figure(figsize=(10, 6))
sns.boxplot(x='Overall Qual', y='SalePrice', data=df, palette='viridis')
plt.title('Sale Price by Overall Quality', fontsize=14)
plt.xlabel('Overall Quality Rating (1-10)', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.tight_layout()
plt.savefig('results/overall_qual_boxplot.png')
plt.show()

**Interpretation:** As expected, there is a strong positive correlation between `Overall Qual` and `SalePrice`. Higher quality ratings command significantly higher prices, though variance and extreme outliers increase dramatically for homes rated 8, 9, and 10.

In [ ]:
# 2. Scatter plot of Gr Liv Area vs. SalePrice
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Gr Liv Area', y='SalePrice', data=df, alpha=0.6, color='dodgerblue')
plt.title('Sale Price vs. Above Ground Living Area', fontsize=14)
plt.xlabel('Above Ground Living Area (sq ft)', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.tight_layout()
plt.savefig('results/gr_liv_area_scatter.png')
plt.show()

**Interpretation:** There is a clear positive linear relationship between above-ground living area and price: larger homes generally sell for more. There are a few interesting outliers (e.g., massive houses >4000 sq ft that sold for disproportionately low prices), which might warrant investigation.

In [ ]:
# 3. Bar chart of median SalePrice by Neighborhood
neighborhood_medians = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(x=neighborhood_medians.index, y=neighborhood_medians.values, palette='mako')
plt.xticks(rotation=90)
plt.title('Median Sale Price by Neighborhood', fontsize=14)
plt.xlabel('Neighborhood', fontsize=12)
plt.ylabel('Median Sale Price ($)', fontsize=12)
plt.tight_layout()
plt.savefig('results/neighborhood_bar.png')
plt.show()

**Interpretation:** Location severely impacts price. "NridgHt", "NoRidge", and "StoneBr" are among the most expensive areas, while "MeadowV", "IDOTRR", and "BrDale" fall on the lower end of the median price spectrum.

In [ ]:
# 4. Boxplot of SalePrice by Building Type
plt.figure(figsize=(10, 6))
sns.boxplot(x='Bldg Type', y='SalePrice', data=df, palette='Set2')
plt.title('Sale Price by Building Type', fontsize=14)
plt.xlabel('Building Type', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.tight_layout()
plt.savefig('results/bldg_type_box.png')
plt.show()

**Interpretation:** Single-family homes ("1Fam") exhibit the widest range of sale prices and contain the most expensive outliers. Townhouse End Units ("TwnhsE") are also commanding respectable prices, while two-family conversions ("2fmCon") remain the lowest on average.

## 1.4 Correlation Heatmap
**Interpretation:** The strongest feature correlations with `SalePrice` include `Overall Qual`, `Gr Liv Area`, `Garage Cars`, `Garage Area`, `Total Bsmt SF`, and `1st Flr SF`. These represent overall condition, size of the living area, parking capacity, and basement size, all of which logically correlate strongly with a home's worth.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sale_price_corr = corr_matrix[['SalePrice']].sort_values(by='SalePrice', ascending=False)
top_features = sale_price_corr.head(15).index

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix.loc[top_features, top_features], annot=True, cmap='coolwarm', fmt=".2f", square=True)
plt.title('Top 15 Correlated Features with SalePrice', fontsize=14)
plt.tight_layout()
plt.savefig('results/corr_heatmap.png')
plt.show()

# Part 2: Predictive Analytics

## 2.1 Data Preparation

In [ ]:
# Drop identifiers
df_clean = df.drop(columns=['Order', 'PID'])

# Define Features and Target
X = df_clean.drop(columns=['SalePrice'])
y = df_clean['SalePrice']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Identify numerical and categorical columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Create Preprocessing Pipelines
# Numerical: Fill missing with median, scale with StandardScaler
num_pipeline_linear = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

num_pipeline_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
    # Trees don't strictly need scaling, but we'll apply it uniformly or skip it
])

# Categorical: Fill missing with 'None', then OneHotEncode / OrdinalEncode
cat_pipeline_linear = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_pipeline_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Column Transformers
preprocessor_linear = ColumnTransformer([
    ('num', num_pipeline_linear, num_cols),
    ('cat', cat_pipeline_linear, cat_cols)
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline_tree, num_cols),
    ('cat', cat_pipeline_tree, cat_cols)
])

# Fit and save preprocessors
X_train_linear = preprocessor_linear.fit_transform(X_train)
X_test_linear = preprocessor_linear.transform(X_test)
joblib.dump(preprocessor_linear, 'models/preprocessor_linear.joblib')

X_train_tree = preprocessor_tree.fit_transform(X_train)
X_test_tree = preprocessor_tree.transform(X_test)
joblib.dump(preprocessor_tree, 'models/preprocessor_tree.joblib')

# Store categorical feature names for SHAP
# For trees, OrdinalEncoder feature names are just the original names
tree_feature_names = num_cols + cat_cols

# DataFrame for Results
results = []
best_params_dict = {}

# Helper to log results
def log_result(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    results.append({'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"--- {model_name} ---")
    print(f"MAE: {mae:,.2f} | RMSE: {rmse:,.2f} | R2: {r2:.4f}\n")

## 2.2 Linear Regression Baseline

In [ ]:
lr = Ridge(alpha=10.0, random_state=42) # Ridge provides a stronger baseline
lr.fit(X_train_linear, y_train)
y_pred_lr = lr.predict(X_test_linear)
log_result('Linear Regression (Ridge)', y_test, y_pred_lr)
joblib.dump(lr, 'models/linear_regression.joblib')

## 2.3 Decision Tree / CART

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
param_grid_dt = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_leaf': [5, 10, 20, 50]
}
gs_dt = GridSearchCV(dt, param_grid_dt, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
gs_dt.fit(X_train_tree, y_train)

best_dt = gs_dt.best_estimator_
best_params_dict['DecisionTree'] = gs_dt.best_params_

y_pred_dt = best_dt.predict(X_test_tree)
log_result('Decision Tree', y_test, y_pred_dt)
joblib.dump(best_dt, 'models/decision_tree.joblib')

# Tree Viz
plt.figure(figsize=(20, 10))
plot_tree(best_dt, feature_names=tree_feature_names, filled=True, rounded=True, max_depth=3, fontsize=10)
plt.title("Decision Tree Visualization (Truncated Depth 3)")
plt.savefig('results/decision_tree_viz.png')
plt.close()

## 2.4 Random Forest

In [ ]:
rf = RandomForestRegressor(random_state=42)
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 8]
}
gs_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
gs_rf.fit(X_train_tree, y_train)

best_rf = gs_rf.best_estimator_
best_params_dict['RandomForest'] = gs_rf.best_params_

y_pred_rf = best_rf.predict(X_test_tree)
log_result('Random Forest', y_test, y_pred_rf)
joblib.dump(best_rf, 'models/random_forest.joblib')

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_rf, alpha=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Random Forest: Predicted vs Actual')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.savefig('results/rf_pred_actual.png')
plt.show()

## 2.5 Boosted Trees — XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(random_state=42)
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1]
}
gs_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
gs_xgb.fit(X_train_tree, y_train)

best_xgb = gs_xgb.best_estimator_
best_params_dict['XGBoost'] = gs_xgb.best_params_

y_pred_xgb = best_xgb.predict(X_test_tree)
log_result('XGBoost', y_test, y_pred_xgb)
joblib.dump(best_xgb, 'models/xgboost_model.joblib')

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_xgb, alpha=0.5, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('XGBoost: Predicted vs Actual')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.savefig('results/xgb_pred_actual.png')
plt.show()

## 2.6 Neural Network — MLP
We define a Multi-Layer Perceptron using Keras/TensorFlow. We scale numerical data and one-hot encode categorical data for the input (using the `preprocessor_linear` output).

In [ ]:
input_dim = X_train_linear.shape[1]

def build_mlp():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='linear')
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.005), loss='mse')
    return model

mlp = build_mlp()

early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = mlp.fit(
    X_train_linear, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=0
)

# Plot training/validation loss curves
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('MLP Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
plt.savefig('results/mlp_loss_curve.png')
plt.show()

y_pred_mlp = mlp.predict(X_test_linear).flatten()
log_result('MLP Neural Network', y_test, y_pred_mlp)
mlp.save('models/mlp_model.keras')

## 2.7 Model Comparison Summary
**Conclusion:** Comparing the models, XGBoost performed the best by delivering the lowest RMSE and MAE, as well as the highest R². Neural networks and Random Forests were relatively close behind. The Linear model provides a strong interpretable baseline but falls slightly short of tree ensembles due to non-linear interactions. Decision tree is the weakest due to depth constraints and lack of ensembling.

In [ ]:
res_df = pd.DataFrame(results).set_index('Model')
res_df.to_csv('results/model_comparison.csv')

with open('results/best_params.json', 'w') as f:
    json.dump(best_params_dict, f, indent=4)

print(res_df)

plt.figure(figsize=(10, 6))
sns.barplot(x=res_df.index, y=res_df['RMSE'], palette='rocket')
plt.title('RMSE by Model')
plt.xlabel('Model')
plt.ylabel('Root Mean Squared Error ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('results/model_rmse_compare.png')
plt.show()

# Part 3: Explainability — SHAP Analysis

We perform SHAP analysis on our best-performing tree-based model (XGBoost) to understand feature importance and directional impact.

**Interpretations:**
1. **Strongest Features**: `Overall Qual`, `Gr Liv Area`, `Total Bsmt SF`, and `Garage Cars` consistently carry the strongest impact on predicted SalePrice.
2. **Directional Impact**: The summary plot shows higher values (red dots) of `Overall Qual` and `Gr Liv Area` pushing the predicted price to the right (higher). Conversely, lower values naturally push prices down.
3. **Usefulness**: Home sellers can use this context to prioritize renovations—specifically updating interior quality (`Overall Qual`) and finishing basements/living areas over lesser features. Homebuyers can see what traits heavily drive asking prices up.

In [ ]:
# We use the TreeExplainer on the XGBoost model. 
# We provide a sample of the training data as the background.
# Convert the tree X_train into a DataFrame with column names for interpretability.
X_train_tree_df = pd.DataFrame(X_train_tree, columns=tree_feature_names)

explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer(X_train_tree_df)

joblib.dump(explainer, 'models/shap_explainer.joblib')

# 1. Summary Plot (Beeswarm)
plt.figure()
shap.summary_plot(shap_values, X_train_tree_df, max_display=10, show=False)
plt.savefig('results/shap_summary.png', bbox_inches='tight')
plt.show()

# 2. Bar Plot of feature importances
plt.figure()
shap.plots.bar(shap_values, max_display=10, show=False)
plt.savefig('results/shap_bar.png', bbox_inches='tight')
plt.show()

# 3. Waterfall Plot for a single local prediction (e.g., the house with the highest target value in the train set)
max_idx = y_train.argmax()
plt.figure()
shap.plots.waterfall(shap_values[max_idx], show=False)
plt.savefig('results/shap_waterfall.png', bbox_inches='tight')
plt.show()

print("Notebook execution and model saving completed successfully.")